# Evaluacion Modelo 9 (Mejor Modelo)

In [3]:
%cd ..

/


In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cv2
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report,accuracy_score
import os
from tensorflow.keras.applications.densenet import preprocess_input

In [6]:
import random
import tensorflow as tf
import os


## Carga de Datos

In [ ]:
from src.carga import cargar_datos

In [ ]:
root = 'data/Garbage-classification/Garbage-classification/'
df = cargar_datos(root)
df.head()

,x,y
1134,data/Garbage-classification/Garbage-classifica...,metal
2487,data/Garbage-classification/Garbage-classifica...,trash
2228,data/Garbage-classification/Garbage-classifica...,plastic
2346,data/Garbage-classification/Garbage-classifica...,plastic
2048,data/Garbage-classification/Garbage-classifica...,plastic


## Carga de datos en Colab

In [9]:
import kagglehub

# Descargar dataset
path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")

print(f"📦 Dataset descargado en: {path}")

Using Colab cache for faster access to the 'garbage-classification' dataset.
📦 Dataset descargado en: /kaggle/input/garbage-classification


In [7]:
import pandas as pd
import os

def cargar_datos_colab(ruta_archivo):
    data = {}

    for i in os.listdir(ruta_archivo):
      if i == 'Garbage classification':
        ruta_clase = os.path.join(ruta_archivo,i)
        if os.path.isdir(ruta_clase):
            for ruta_actual, subcarpetas, archivos in os.walk(ruta_clase):
                for k in archivos:
                    data[os.path.join(ruta_actual, k)] = os.path.basename(ruta_actual)

    df = pd.DataFrame(data.items(), columns=['x', 'y'])
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

In [10]:
df = cargar_datos_colab(path)

## Division de los datos

In [11]:
from sklearn.model_selection import train_test_split

# 80% train, 20% (test + validación)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['y']
 )

# Del 20% (test + validacion): 10% validación y 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['y']
 )

print(f"Train: {len(train_df)} ({len(train_df)/len(df):.1%})")
print(f"Validación: {len(val_df)} ({len(val_df)/len(df):.1%})")
print(f"Test: {len(test_df)} ({len(test_df)/len(df):.1%})")

Train: 2021 (80.0%)
Validación: 253 (10.0%)
Test: 253 (10.0%)


## Modificacion de las imagenes y carga en memoria.

In [12]:
def load_images(df, size):
    images = []
    labels = []

    for _, row in df.iterrows():
        ruta = row['x']
        label = row['y']

        img = cv2.imread(ruta)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (size, size))
        #img = img.astype('float32') / 255.0

        images.append(img)
        labels.append(label)

    X = np.array(images)
    y = np.array(labels)

    return X, y

In [13]:
x_train, y_train = load_images(train_df, size=128)
x_val, y_val = load_images(val_df, size=128)
x_test, y_test = load_images(test_df, size=128)

In [14]:

x_train = preprocess_input(x_train.astype("float32"))
x_val = preprocess_input(x_val.astype("float32"))
x_test = preprocess_input(x_test.astype("float32"))

In [15]:
x_train.shape, y_train.shape, x_val.shape, y_val.shape, x_test.shape, y_test.shape

((2021, 128, 128, 3),
 (2021,),
 (253, 128, 128, 3),
 (253,),
 (253, 128, 128, 3),
 (253,))

Encoder para los labels

In [16]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val  = le.transform(y_val)
y_test  = le.transform(y_test)

In [17]:
le.classes_

array(['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash'],
      dtype='<U9')

## Aplicacion del modelo

In [18]:
from modelo9 import estructura_modelo
from modelo_simple import compilar
from curvas import graf_pedida,graf_acc,plot_confusion
from semillas import aplicar_semilla

## Bucle para probar rendimiento modelo con diferentes semillas

In [19]:
semillas = [42, 123, 7, 99]
resultados = {}


for i in semillas:
    print(f"Entrenando modelo con semilla: {i}")
    aplicar_semilla(i)

   #Defunimos arquitectura y compilamos el modelo
    model = estructura_modelo(input_shape=(128, 128, 3), num_classes=len(le.classes_))#,use_augmentation=True)
    model = compilar(model,learning_rate=1e-4)

    #Definimos callbacks
    ReduceLROnPlateau=tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=6,
        min_lr=1e-6,
        verbose=1
    )
    EarlyStopping=tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=20,
            restore_best_weights=True,
            verbose=1
        )


    #Ponemos a entrenar el modelo
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=150,
        batch_size=128,
        callbacks=[ReduceLROnPlateau,EarlyStopping]
    )

    train = model.evaluate(x_train, y_train, verbose=0,steps=100)
    val = model.evaluate(x_val, y_val, verbose=0,steps=100)
    test = model.evaluate(x_test, y_test, verbose=0,steps=100)
    resultados[i] = {'train': train, 'val': val, 'test': test}

    #graf_acc(history, save_path=f"reports/semillas/acc/acc_seed_{i}.png")
    #graf_pedida(history, save_path=f"reports/semillas/loss/loss_seed_{i}.png")


    #y_pred = np.argmax(model.predict(x_test), axis=1)
    #plot_confusion(y_test, y_pred, le.classes_, i,save_path=f"reports/semillas/conf_matrix/confusion_seed_{i}.png")

    #print(f"Classification Report{i} - test:")
    #print(classification_report(y_test, y_pred, target_names=le.classes_))

Entrenando modelo con semilla: 42


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,349,574 (28.04 MB)

 Trainable params: 7,265,926 (27.72 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │       262,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_10 (Activation)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_11 (Activation)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,349,574 (28.04 MB)

 Trainable params: 7,265,926 (27.72 MB)

 Non-trainable params: 83,648 (326.75 KB)

Epoch 1/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 276s 8s/step - loss: 2.1962 - sparse_categorical_accuracy: 0.2291 - val_loss: 1.4891 - val_sparse_categorical_accuracy: 0.4308 - learning_rate: 1.0000e-04
Epoch 2/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 373ms/step - loss: 1.6987 - sparse_categorical_accuracy: 0.3483 - val_loss: 1.2229 - val_sparse_categorical_accuracy: 0.5692 - learning_rate: 1.0000e-04
Epoch 3/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 386ms/step - loss: 1.4362 - sparse_categorical_accuracy: 0.4344 - val_loss: 1.0487 - val_sparse_categorical_accuracy: 0.6443 - learning_rate: 1.0000e-04
Epoch 4/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 383ms/step - loss: 1.2216 - sparse_categorical_accuracy: 0.5334 - val_loss: 0.8686 - val_sparse_categorical_accuracy: 0.6877 - learning_rate: 1.0000e-04
Epoch 5/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 397ms/step - loss: 1.0439 - sparse_categorical_accuracy: 0.6160 - val_loss: 0.7516 - val_sparse_categorical_accuracy: 0.7115 - learning_rate: 1.0000e-04
Epoch 6/150
16/16 ━━━━━━━━

In [20]:
# Extrae solo el accuracy (índice 1) de cada lista
resultados_clean = {
    seed: {split: valores[1] for split, valores in metricas.items()}
    for seed, metricas in resultados.items()
}

df_resultados = pd.DataFrame(resultados_clean).T
df_resultados.index.name = 'Seed'
df_resultados.columns = ['Train Accuracy', 'Val Accuracy', 'Test Accuracy']
df_resultados.loc['Media'] = df_resultados.mean()

print(df_resultados.round(4))

       Train Accuracy  Val Accuracy  Test Accuracy
Seed                                              
42              0.999        0.8972         0.9012
123             0.999        0.8893         0.8854
7               0.999        0.8893         0.8696
99              0.999        0.8933         0.8972
Media           0.999        0.8923         0.8883


In [21]:
!pip install tabulate

In [22]:
print(df_resultados.round(4).to_markdown())

| Seed   |   Train Accuracy |   Val Accuracy |   Test Accuracy |
|:-------|-----------------:|---------------:|----------------:|
| 42     |            0.999 |         0.8972 |          0.9012 |
| 123    |            0.999 |         0.8893 |          0.8854 |
| 7      |            0.999 |         0.8893 |          0.8696 |
| 99     |            0.999 |         0.8933 |          0.8972 |
| Media  |            0.999 |         0.8923 |          0.8883 |
